# DiD-BCF — B1_baseline (linearity_degree=1)

**Workstream B1 · canonical DiD (selection on unobservables)**

moderate unit FE, selection on unobservables, iid errors

Fits DiD-BCF on the `B1_baseline` scenario at `linearity_degree=1` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.3 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_baseline",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_baseline_lin_1] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_baseline: 100%|██████████| 100/100 [6:46:07<00:00, 243.67s/fit]

[B1_baseline_lin_1] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_baseline_lin_1.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_baseline,1,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.290667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
1,canonical,B1_baseline,1,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.316667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
2,canonical,B1_baseline,1,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.437333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
3,canonical,B1_baseline,1,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.462667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
4,canonical,B1_baseline,1,200,0,ES,k=0,NaN,NaN,0.0,...,0.290667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_baseline,1,200,ATT,ATT,power,-0.625652,-3.744373,0.31,...,0.758931,1.481579,0.680558,3.744373,1.0,0.0,0.844670,3.746017,0.337683,5.497713
1,canonical,B1_baseline,1,200,ES,k=0,power,-0.635033,-3.753852,0.36,...,0.933714,1.350285,0.696126,3.753852,1.0,0.0,0.859451,3.755049,0.406154,26.790424
2,canonical,B1_baseline,1,200,ES,k=1,power,-0.623022,-3.746290,0.38,...,0.919593,1.529049,0.673820,3.746290,1.0,0.0,0.836728,3.748242,0.415246,4.900630
3,canonical,B1_baseline,1,200,ES,k=2,power,-0.623782,-3.739901,0.36,...,0.943363,1.649129,0.689336,3.739901,1.0,0.0,0.854792,3.742116,0.407555,4.157410
4,canonical,B1_baseline,1,200,ES,k=3,power,-0.620771,-3.737450,0.33,...,0.932340,1.758227,0.685970,3.737450,1.0,0.0,0.848928,3.739902,0.405850,4.134825
5,canonical,B1_baseline,1,200,GATT,g=4_t=4,power,-0.635033,-3.753852,0.36,...,0.933714,1.350285,0.696126,3.753852,1.0,0.0,0.859451,3.755049,0.406154,26.790424
6,canonical,B1_baseline,1,200,GATT,g=4_t=5,power,-0.623022,-3.746290,0.38,...,0.919593,1.529049,0.673820,3.746290,1.0,0.0,0.836728,3.748242,0.415246,4.900630
7,canonical,B1_baseline,1,200,GATT,g=4_t=6,power,-0.623782,-3.739901,0.36,...,0.943363,1.649129,0.689336,3.739901,1.0,0.0,0.854792,3.742116,0.407555,4.157410
8,canonical,B1_baseline,1,200,GATT,g=4_t=7,power,-0.620771,-3.737450,0.33,...,0.932340,1.758227,0.685970,3.737450,1.0,0.0,0.848928,3.739902,0.405850,4.134825


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_baseline,1,200,corrected,100,401.24,1.181223,0.310973,0.996955,...,0.256580,0.056486,0.233734,0.067449,0.27708,0.07856,0.777697,0.084622,0.932253,0.106298
1,canonical,B1_baseline,1,200,plain,100,401.24,3.845845,0.109213,3.744373,...,0.996139,0.017973,0.000000,0.000000,0.00000,0.00000,1.251232,0.066233,1.577289,0.080679
